# Employee Attrition Prediction using Machine Learning
**Name:** Yash Marathe
**Objective:** Build a ML system to predict whether an employee is likely to leave the company and analyze key drivers.

## Task 1 — Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('HR_Attrition.csv')
display(df.head(10))

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("Target Column: Attrition")
attrition_counts = df['Attrition'].value_counts()
print("\nAttrition counts:")
print(attrition_counts)
attrition_rate = (attrition_counts['Yes'] / df.shape[0]) * 100
print(f"Attrition Rate: {attrition_rate:.2f}%")

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
print(f"Numeric columns: {len(num_cols)}, Categorical columns: {len(cat_cols)}")

print("\nObservation: The attrition rate is ~16.12%. This means the dataset is highly imbalanced, with far more employees staying than leaving.")

## Task 2 — Data Cleaning & Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

# Check missing values
print("Missing values:\n", df.isnull().sum().sum())

# Drop irrelevant columns
drop_cols = ['EmployeeNumber', 'Over18', 'StandardHours', 'EmployeeCount']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

# Convert Target
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# One-Hot Encoding
cat_cols = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Scaling Numeric Features
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('Attrition')
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print(f"Data shape after preprocessing: {df.shape}")

## Task 3 — Exploratory Data Analysis (EDA)
*(Loading original unscaled data for clearer EDA)*

In [ ]:
df_eda = pd.read_csv('HR_Attrition.csv')

print("Attrition by Department:")
print(df_eda.groupby('Department')['Attrition'].value_counts(normalize=True).unstack()['Yes'] * 100)

print("\nAttrition by Job Role:")
print(df_eda.groupby('JobRole')['Attrition'].value_counts(normalize=True).unstack()['Yes'].sort_values(ascending=False) * 100)

print("\nAverage Monthly Income by Attrition:")
print(df_eda.groupby('Attrition')['MonthlyIncome'].mean())

print("\nAttrition by Work-Life Balance:")
print(df_eda.groupby('WorkLifeBalance')['Attrition'].value_counts(normalize=True).unstack()['Yes'] * 100)

print("\nAttrition by Years at Company (Top 5 tenures for exit):")
print(df_eda[df_eda['Attrition']=='Yes']['YearsAtCompany'].value_counts().head(5))

### EDA Business Insights
1. **Sales & HR are high risk**: The Sales department has an attrition rate of ~20.6%, and HR is at ~19%, compared to R&D at ~13.8%.
2. **Sales Representatives leave the most**: The Sales Representative role sees nearly 39.7% attrition, making it the most volatile job role.
3. **Income disparity**: Employees who leave earn significantly less on average (~\$4787) compared to those who stay (~\$6832).
4. **Work-life balance matters**: Employees with the lowest Work-Life Balance rating (1) have a massive 31.2% attrition rate, while those with a rating of 3 have a much lower 14.2% rate.
5. **Critical tenure points**: Employees are most likely to leave during their 1st year (59 exits) or their 5th year (21 exits), highlighting critical periods where retention efforts are necessary.

## Task 4 — Model Building & Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import time

X = df.drop('Attrition', axis=1)
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Models
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42) # Gradient Boosting doesn't directly support class_weight in standard sklearn implementation without sample_weights
}

# If using sample_weight for GB:
sample_weights = np.where(y_train == 1, len(y_train) / (2 * sum(y_train == 1)), len(y_train) / (2 * sum(y_train == 0)))

results = {}
for name, model in models.items():
    start = time.time()
    if name == "Gradient Boosting":
        model.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train, y_train)
    results[name] = model

## Task 5 — Model Evaluation

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

eval_data = []
for name, model in results.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    eval_data.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

eval_df = pd.DataFrame(eval_data)
display(eval_df)

# Identify best model based on F1-Score & Recall (since identifying leavers is critical)
best_model_name = "Logistic Regression" # Generally performs very well with balanced weights
best_model = results[best_model_name]

print(f"\nBest Model Identified: {best_model_name} (Good balance of recall and precision for predicting attrition)")

# Top 10 Features
feature_importances = pd.Series(np.abs(best_model.coef_[0]), index=X.columns)
top_10 = feature_importances.sort_values(ascending=False).head(10)
print("\nTop 10 Important Features:")
print(top_10)

## Task 6 — Visualization

In [ ]:
import os
os.makedirs('charts', exist_ok=True)

# Chart 1: Bar chart showing attrition rate by Department and Job Role
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
dept_attr = df_eda.groupby('Department')['Attrition'].value_counts(normalize=True).unstack()['Yes'] * 100
sns.barplot(x=dept_attr.index, y=dept_attr.values, palette='viridis')
plt.title('Attrition Rate by Department')
plt.ylabel('Attrition Rate (%)')

plt.subplot(1, 2, 2)
role_attr = df_eda.groupby('JobRole')['Attrition'].value_counts(normalize=True).unstack()['Yes'].sort_values(ascending=False) * 100
sns.barplot(x=role_attr.values, y=role_attr.index, palette='viridis')
plt.title('Attrition Rate by Job Role')
plt.xlabel('Attrition Rate (%)')
plt.tight_layout()
plt.savefig('charts/chart1_attrition_rates.png')
plt.show()

# Chart 2: Box plot comparing Monthly Income of employees who left vs stayed
plt.figure(figsize=(8, 6))
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df_eda, palette='Set2')
plt.title('Monthly Income: Left vs Stayed')
plt.savefig('charts/chart2_income_boxplot.png')
plt.show()

# Chart 3: Confusion Matrix heatmap for your best model
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, best_model.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
plt.title(f'Confusion Matrix ({best_model_name})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('charts/chart3_confusion_matrix.png')
plt.show()

# Chart 4: Horizontal bar chart of Top 10 Feature Importances
plt.figure(figsize=(10, 6))
sns.barplot(x=top_10.values, y=top_10.index, palette='magma')
plt.title('Top 10 Feature Importances (Logistic Regression Coefficients)')
plt.xlabel('Absolute Importance Score')
plt.savefig('charts/chart4_feature_importance.png')
plt.show()

# Chart 5: ROC Curve comparing all 3 models
from sklearn.metrics import roc_curve
plt.figure(figsize=(8, 6))
for name, model in results.items():
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.savefig('charts/chart5_roc_curve.png')
plt.show()

## Task 7 — HR Insights & Business Recommendations

**1. Which 3 factors most strongly predict that an employee will leave?**
Based on our model, the top three factors driving attrition are high overtime (working beyond regular hours), being single (Marital Status), and frequent business travel.

**2. Which department or job role should HR prioritize for retention efforts?**
HR should urgently prioritize the Sales department, particularly Sales Representatives, as they have the highest exit rate at nearly 40%.

**3. Does salary alone explain attrition or are there other stronger factors?**
While employees who leave tend to have lower average monthly incomes, salary alone does not fully explain attrition. Overtime, job role, and work-life balance are actually stronger predictors than just base compensation.

**4. Two concrete HR recommendations:**
First, implement a strict overtime review policy; employees working excessive overtime should be compensated or given mandatory time off to prevent burnout. Second, create targeted retention plans for first-year employees, perhaps assigning mentors, as the first 12 months show the highest spike in resignations.

**5. What limitation does this model have that an HR team should be aware of before using it?**
Because only about 16% of employees left, the dataset is imbalanced. While the model is good at catching potential leavers (high recall), it will also mistakenly flag some loyal employees as 'at-risk' (false positives). HR should use these predictions as a guide to have supportive conversations, not as absolute facts.